In [12]:
# Install PyTorch Geometric for Graph Neural Network operations
!pip install torch-geometric

In [13]:
import pickle
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax, dropout_edge

# Set device to GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# --- Hyperparameters ---
EMBED_DIM = 200        # Dimension of entity and relation embeddings
NUM_LAYERS = 2         # Number of GNN layers
NUM_HEADS = 4          # Number of attention heads
BATCH_SIZE = 1024      # Training batch size
LR = 0.001             # Learning rate
EPOCHS = 200           # Number of training epochs
DROPOUT = 0.1          # Standard dropout rate
edge_dropout_rate = 0.1 # Dropout rate for edges during training
WEIGHT_DECAY = 1e-4    # L2 regularization penalty

# Dimensions for the ConvE decoder reshaping (200 = 10 * 20)
CONV_DW = 10
assert EMBED_DIM % CONV_DW == 0
CONV_DH = EMBED_DIM // CONV_DW

Device: cuda


In [14]:
# Define paths to the dataset
base_path = "/kaggle/input/fb13k-237/" 
train_path = f"{base_path}train.txt"
valid_path = f"{base_path}valid.txt"
test_path = f"{base_path}test.txt"

column_names = ["head", "relation", "tail"]

# Load train, validation, and test splits into pandas DataFrames
df_train = pd.read_csv(train_path, sep="\t", header=None, names=column_names)
df_valid = pd.read_csv(valid_path, sep="\t", header=None, names=column_names)
df_test  = pd.read_csv(test_path,  sep="\t", header=None, names=column_names)

print(f"Train: {len(df_train)} Valid: {len(df_valid)} Test: {len(df_test)}")

Train: 272115 Valid: 17535 Test: 20466


In [15]:
# Extract unique entities and relations across all splits
entities = set(df_train["head"]) | set(df_train["tail"]) | \
           set(df_valid["head"]) | set(df_valid["tail"]) | \
           set(df_test["head"])  | set(df_test["tail"])

relations = set(df_train["relation"]) | set(df_valid["relation"]) | set(df_test["relation"])

# Create mapping dictionaries from string to ID
ent2id = {e:i for i,e in enumerate(sorted(entities))}
rel2id = {r:i for i,r in enumerate(sorted(relations))}

num_entities = len(ent2id)
original_num_relations = len(rel2id)

# Double relations to account for inverse relations (t, r_inv, h)
num_relations = original_num_relations * 2 

print("Entities:", num_entities)
print("Original Relations:", original_num_relations)
print("Total Relations (with inverses):", num_relations)

# Helper function to convert text DataFrames to integer ID triples
def df_to_ids(df):
    triples = []
    for _, row in df.iterrows():
        h = ent2id[row["head"]]
        r = rel2id[row["relation"]]
        t = ent2id[row["tail"]]
        triples.append((h,r,t))
    return triples

# Convert all splits to ID format
train_ids_orig = df_to_ids(df_train)
valid_ids_orig = df_to_ids(df_valid)
test_ids_orig  = df_to_ids(df_test)

train_ids = [] 
src_list = []
dst_list = []
rel_list = []

# Build bidirectional graph data for PyG
for h, r, t in train_ids_orig:
    # Forward edge
    train_ids.append((h, r, t))
    src_list.append(h)
    dst_list.append(t)
    rel_list.append(r)
    
    # Inverse edge
    r_inv = r + original_num_relations
    train_ids.append((t, r_inv, h))
    src_list.append(t)
    dst_list.append(h)
    rel_list.append(r_inv)

# Convert lists to PyTorch tensors for PyG Message Passing
edge_index = torch.tensor([src_list, dst_list], dtype=torch.long, device=DEVICE)
edge_type = torch.tensor(rel_list, dtype=torch.long, device=DEVICE)

# Dictionary mapping (head, relation) -> set of valid tails (for filtering during eval)
hr2t = defaultdict(set)
all_original_triples = train_ids_orig + valid_ids_orig + test_ids_orig

for h, r, t in all_original_triples:
    hr2t[(h, r)].add(t)                     # Forward lookup
    hr2t[(t, r + original_num_relations)].add(h) # Inverse lookup

hr2t = dict(hr2t)

print("Bidirectional Graph tensors constructed:", edge_index.shape)

Entities: 14541
Original Relations: 237
Total Relations (with inverses): 474
Bidirectional Graph tensors constructed: torch.Size([2, 544230])


In [16]:
# Group train tails by (head, relation) to train with 1-N scoring (multi-hot targets)
hr2tails_train = defaultdict(list)
for h, r, t in train_ids:
    hr2tails_train[(h, r)].append(t)

unique_train_hr = list(hr2tails_train.keys())

# Dataset class that yields a head, relation, and all its true tails
class MultiHotKGDataset(Dataset):
    def __init__(self, unique_hr, hr2tails):
        self.unique_hr = unique_hr
        self.hr2tails = hr2tails

    def __len__(self):
        return len(self.unique_hr)

    def __getitem__(self, idx):
        h, r = self.unique_hr[idx]
        tails = self.hr2tails[(h, r)]
        return h, r, tails

# Collate function to format batches into multi-hot probability distributions
def ce_prob_collate(batch):
    hs, rs, tails_list = zip(*batch)
    
    hs = torch.tensor(hs, dtype=torch.long)
    rs = torch.tensor(rs, dtype=torch.long)
    
    # Initialize zero target matrix
    targets = torch.zeros(len(batch), num_entities, dtype=torch.float32)
    
    # Distribute probability mass equally among all true tails
    for i, tails in enumerate(tails_list):
        prob_mass = 1.0 / len(tails)
        targets[i, tails] = prob_mass
        
    return hs, rs, targets

# Create the training dataloader
train_loader = DataLoader(
    MultiHotKGDataset(unique_train_hr, hr2tails_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ce_prob_collate,
    num_workers=4,            
    pin_memory=True,           
    persistent_workers=True,   
    prefetch_factor=2          
)

In [17]:
# Custom Graph Attention layer with relation embeddings
class GATLayer(MessagePassing):
    def __init__(self, dim, num_heads=4):
        super().__init__(aggr='add', flow='source_to_target', node_dim=0)
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        
        # Projections for Query, Key, Value
        self.Wq = nn.Linear(dim, dim)
        self.Wk = nn.Linear(dim, dim)
        self.Wv = nn.Linear(dim, dim)
        
        # Attention scorer
        self.att_proj = nn.Linear(self.head_dim, 1)
        
        # Relation integration projection
        self.W_rel = nn.Linear(dim, dim)
        
        # Output transformations
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, H, R, edge_index, edge_type):
        # Compute Q, K, V for all nodes
        Q = self.Wq(H).view(-1, self.num_heads, self.head_dim)
        K = self.Wk(H).view(-1, self.num_heads, self.head_dim)
        V = self.Wv(H).view(-1, self.num_heads, self.head_dim)
        
        # Compute relation embeddings for each edge
        r_emb = self.W_rel(R[edge_type]).view(-1, self.num_heads, self.head_dim)
        
        # Start message passing
        out = self.propagate(edge_index, Q=Q, K=K, V=V, r_emb=r_emb)
        
        # Final linear projection, dropout, residual connection, and norm
        out = out.reshape(-1, self.dim)
        out = self.dropout(self.out_proj(out))
        
        return self.norm(H + self.act(out))

    def message(self, Q_i, K_j, V_j, r_emb, index, ptr, size_i):
        # Incorporate relation embedding into Key
        K_combined = K_j + r_emb
        
        # Calculate attention scores
        attn_score = self.att_proj(Q_i * K_combined) 
        e = F.leaky_relu(attn_score, negative_slope=0.2)
        
        # Softmax over neighborhood and dropout
        attn_weights = softmax(e, index, ptr, size_i)
        attn_weights = F.dropout(attn_weights, p=DROPOUT, training=self.training)

        # Incorporate relation embedding into Value and weight by attention
        V_combined = V_j + r_emb
        return attn_weights * V_combined
      
# The full encoder chaining multiple GATLayers  
class GATEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Initial embeddings
        self.ent = nn.Embedding(num_entities, EMBED_DIM)
        self.rel = nn.Embedding(num_relations, EMBED_DIM)
        
        self.layers = nn.ModuleList([
            GATLayer(EMBED_DIM, NUM_HEADS) for _ in range(NUM_LAYERS)
        ])
        
        # Initialize weights
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)

    def forward(self, edge_index, edge_type):
        H = self.ent.weight
        R = self.rel.weight
        
        # Pass through GNN layers
        for layer in self.layers:
            H = layer(H, R, edge_index, edge_type)
            
        return H, R

In [18]:
# ConvE Decoder to score (head, relation) against all tails
class ConvEDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 2D Convolution layers
        self.conv = nn.Conv2d(1, 32, (3, 3), padding=0) 
        self.bn0 = nn.BatchNorm2d(1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm1d(EMBED_DIM)
        
        # Dropouts
        self.input_drop = nn.Dropout(0.2)       
        self.feature_map_drop = nn.Dropout2d(0.2) 
        self.hidden_drop = nn.Dropout(0.3)      
        
        # Calculate flattened dimension size after convolution
        h_out = (2 * CONV_DH) - 2
        w_out = CONV_DW - 2
        self.flat_sz = 32 * h_out * w_out
        
        self.fc = nn.Linear(self.flat_sz, EMBED_DIM)
        
        # Entity bias for final scoring
        self.entity_bias = nn.Parameter(torch.zeros(num_entities))

    def forward(self, h, r, E):
        B = h.size(0)
        
        # Reshape 1D embeddings into 2D maps
        h = h.view(-1, 1, CONV_DH, CONV_DW)
        r = r.view(-1, 1, CONV_DH, CONV_DW)
        
        # Concatenate head and relation maps vertically
        inputs = torch.cat([h, r], dim=2)
        
        # Optimize memory format
        inputs = inputs.to(memory_format=torch.channels_last)
        
        # Convolutions
        x = self.bn0(inputs)
        x = self.input_drop(x)        
        
        x = self.conv(x)              
        x = self.bn1(x)
        x = F.relu(x)
        x = self.feature_map_drop(x)  
        
        # Flatten and project back to EMBED_DIM
        x = x.flatten(1)
        
        x = self.fc(x)
        x = self.hidden_drop(x)       
        x = self.bn2(x)
        x = F.relu(x)
        
        # Dot product with all entity embeddings + bias
        scores = (x @ E.t()) + self.entity_bias
        
        return scores

# Combine Encoder and Decoder
class GATCE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = GATEncoder()
        self.dec = ConvEDecoder()

In [19]:
# Collate function for evaluation (creates masks for filtered metrics)
def eval_collate(batch):
    hs, rs, ts = zip(*batch)
    
    hs = torch.tensor(hs, dtype=torch.long)
    rs = torch.tensor(rs, dtype=torch.long)
    ts = torch.tensor(ts, dtype=torch.long)
    
    # Mask to filter out known positive tails
    mask = torch.zeros((len(batch), num_entities), dtype=torch.bool)
    
    for i, (h, r, t) in enumerate(batch):
        # Fetch all known valid tails for this (h, r)
        known_tails = list(hr2t[(h, r)])
        
        mask[i, known_tails] = True # Mask all true known tails
        mask[i, t] = False          # Unmask the specific target tail for this query
        
    return hs, rs, ts, mask

# Simple dataset wrapper for evaluation queries
class EvalDataset(Dataset):
    def __init__(self, triples):
        self.triples = triples
  
    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        return self.triples[idx]

# Function to compute MRR and Hits@K metrics
def evaluate(model, loader):
    model.eval()
    
    # Precompute structural embeddings once
    with torch.no_grad():
        H_full, R_full = model.enc(edge_index, edge_type)
    
    hits1, hits3, hits10, mrr, total = 0, 0, 0, 0, 0
    
    with torch.no_grad():
        for h, r, t, mask in tqdm(loader, desc="Evaluating", leave=False):
           
            h, r, t = h.to(DEVICE), r.to(DEVICE), t.to(DEVICE)
            mask = mask.to(DEVICE) 
            
            batch_h = H_full[h]
            batch_r = R_full[r]
            
            # Get scores for all entities
            scores = model.dec(batch_h, batch_r, H_full) 
            
            # Apply filtered mask (sets known false positives to -inf)
            scores.masked_fill_(mask, -1e9)
                    
            # Add tiny noise to break exact ties randomly
            noise = torch.randn_like(scores) * 1e-10
            scores += noise
            
            # Calculate rank of true tail
            true_scores = scores.gatceer(1, t.view(-1, 1))
            rank = (scores > true_scores).sum(dim=1).float() + 1

            # Accumulate metrics
            mrr += (1.0 / rank).sum().item()
            hits1 += (rank <= 1).sum().item()
            hits3 += (rank <= 3).sum().item()
            hits10 += (rank <= 10).sum().item()
            total += len(h)
            
    return {
        "MRR": mrr / total,
        "Hits@1": hits1 / total,
        "Hits@3": hits3 / total,
        "Hits@10": hits10 / total
    }

In [20]:
if __name__ == '__main__':

    # Prepare validation queries (forward and inverse)
    valid_queries = []
    for h, r, t in valid_ids_orig: 
        valid_queries.append((h, r, t))
        valid_queries.append((t, r + original_num_relations, h))
        
    valid_loader = DataLoader(
        EvalDataset(valid_queries), 
        batch_size=64, 
        num_workers=4,           
        pin_memory=True,         
        collate_fn=eval_collate
    )

    # Initialize model
    model = GATCE().to(DEVICE)
    
    # Setup optimizer, scheduler, loss, and mixed precision scaler
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) 
    scaler = torch.amp.GradScaler('cuda')

    eval_every = 10
    best_mrr = 0.0

    print(f"Starting training. Evaluating every {eval_every} epochs...")

    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

        for batch in pbar:
            h_idx, r_idx, targets = batch
            h_idx, r_idx, targets = h_idx.to(DEVICE), r_idx.to(DEVICE), targets.to(DEVICE)
            
            # Use this function to get the dropped indices and the retention mask
            dropped_edge_index, edge_mask = dropout_edge(
                edge_index, 
                p=edge_dropout_rate, 
                force_undirected=False, 
                training=True
            )
            
            # Apply the mask to your edge_type tensor manually
            dropped_edge_type = edge_type[edge_mask]
            
            opt.zero_grad()
            
            # Forward pass with mixed precision
            with torch.amp.autocast('cuda'):
                H_full, R_full = model.enc(dropped_edge_index, dropped_edge_type)
                batch_h = H_full[h_idx]
                batch_r = R_full[r_idx]
                scores = model.dec(batch_h, batch_r, H_full)
                loss = loss_fn(scores, targets)

            # Backward pass & Optimizer Step
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)  
            scaler.update()

            total_loss += loss.item()
            pbar.set_postfix({"loss": total_loss/(pbar.n+1)})
        
        # Validation
        if epoch % eval_every == 0:
            metrics = evaluate(model, valid_loader)
            current_mrr = metrics["MRR"]
            current_lr = opt.param_groups[0]['lr']
            
            print(f"Epoch {epoch} | Loss: {total_loss/len(train_loader):.4f} | Val MRR: {current_mrr:.4f} | H@10: {metrics['Hits@10']:.4f}")

            sched.step(current_mrr)
                       
            # Save the best model state
            if current_mrr > best_mrr:
                best_mrr = current_mrr
                print(f"New best MRR! Saving checkpoint...")
                torch.save(model.state_dict(), "best_gatce_model.pth")
                
    print(f"Training complete. Best Validation MRR: {best_mrr:.4f}")

Starting training. Evaluating every 10 epochs...


Epoch 1:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/147 [00:00<?, ?it/s]

...

Epoch 198:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 199:   0%|          | 0/147 [00:00<?, ?it/s]

Epoch 200:   0%|          | 0/147 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/548 [00:00<?, ?it/s]

Epoch 200 | Loss: 2.4399 | Val MRR: 0.2930 | H@10: 0.4591
Training complete. Best Validation MRR: 0.2941


In [22]:
# Prepare test queries (forward and inverse)
test_queries = []
for h, r, t in test_ids_orig:
    test_queries.append((h, r, t))
    test_queries.append((t, r + original_num_relations, h))

print(f"Loading best checkpoint for final evaluation on Test Set ({len(test_queries)} queries)...")

# Load best model checkpoint
best_model = GATCE().to(DEVICE)
best_model.load_state_dict(torch.load("best_gatce_model.pth"))

test_loader = DataLoader(
    EvalDataset(test_queries), 
    batch_size=64, 
    num_workers=4,           
    pin_memory=True,         
    collate_fn=eval_collate
)

# Run test evaluation
test_metrics = evaluate(best_model, test_loader)

print("\n=== FINAL TEST RESULTS ===")
print(f"MRR:     {test_metrics['MRR']:.4f}")
print(f"Hits@1:  {test_metrics['Hits@1']:.4f}")
print(f"Hits@3:  {test_metrics['Hits@3']:.4f}")
print(f"Hits@10: {test_metrics['Hits@10']:.4f}")

# Save the entity and relation ID mappings to use for inference later
mappings = {
    "ent2id": ent2id,
    "rel2id": rel2id,
    "id2ent": {v: k for k, v in ent2id.items()},
    "id2rel": {v: k for k, v in rel2id.items()}
}

with open("mappings.pkl", "wb") as f:
    pickle.dump(mappings, f)

print("Mappings saved successfully.")

Loading best checkpoint for final evaluation on Test Set (40932 queries)...


Evaluating:   0%|          | 0/640 [00:00<?, ?it/s]


=== FINAL TEST RESULTS ===
MRR:     0.2883
Hits@1:  0.2062
Hits@3:  0.3152
Hits@10: 0.4556
Mappings saved successfully.
